# EmbedPlan quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/embedplan/EmbedPlan/blob/main/examples/quickstart.ipynb)

EmbedPlan is a transition model for planning: given a **state** and an **action** written as text, it predicts the **next state**. It embeds both with a frozen text encoder, predicts the next state's embedding with a small learned network, and returns the closest real state. Paper: [Textual Planning with Explicit Latent Transitions](https://arxiv.org/abs/2602.04557).

This notebook runs on a CPU in a few minutes and needs no download: it uses a toy ferry world and a word n-gram encoder. Swap in your own transitions and a stronger encoder at the end.

In [ ]:
# In Colab (or any fresh environment), install the package from GitHub:
%pip install -q git+https://github.com/embedplan/EmbedPlan.git

## 1. Data: transitions as text

A transition is `(state, action) -> next_state`. `groups` says which problem each transition comes from, so we can test on problems the model never saw.

In [ ]:
from embedplan import EmbedPlan, split_transitions, transitions_from_trajectories
from embedplan.datasets import load_toy_ferry

data = load_toy_ferry()
print(len(data.X), "transitions from", len(set(data.groups)), "problems\n")
state, action = data.X[0]
print("state :", state)
print("action:", action)
print("next  :", data.y[0])

## 2. Train and evaluate under the paper's two main protocols

- **Interpolation**: held-out transitions of problems the model has seen.
- **Extrapolation**: whole problems held out, so every test state is new.

`evaluate` ranks each true next state among up to 128 candidates and reports the chance level of the same pools, so every number can be read against chance.

In [ ]:
models = {}
for protocol in ("interpolation", "extrapolation"):
    X_tr, X_te, y_tr, y_te, g_tr, g_te = split_transitions(data.X, data.y, data.groups, protocol=protocol)
    grouped = protocol == "extrapolation"
    model = EmbedPlan(encoder="hashing").fit(X_tr, y_tr, groups=g_tr if grouped else None)
    r = model.evaluate(X_te, y_te, groups=g_te if grouped else None)
    models[protocol] = (model, X_te, y_te, g_te)
    print(f"{protocol:>13}: Hit@1 {r['hit@1']:.2f}  Hit@5 {r['hit@5']:.2f}  "
          f"(chance@5 {r['chance@5']:.2f}, {r['mean_pool_size']:.0f} candidates)")


As in the paper, accuracy is high on problems the model has seen and drops on unseen ones, while staying above chance.

## 3. Predict and roll out a plan

`predict` returns the closest **known** state.


In [ ]:
# On problems the model has seen: predict one step, then roll out a whole plan.
model = models["interpolation"][0]
run = data.frame[(data.frame.problem == 0) & (data.frame.plan == 0)]
print("predicted:", model.predict([(run.state.iloc[0], run.action.iloc[0])])[0][:90], "...")
print("true     :", run.next_state.iloc[0][:90], "...")
rolled = model.rollout(run.state.iloc[0], run.action.tolist())
print(f"rollout: {sum(p == t for p, t in zip(rolled, run.next_state))} of {len(run)} steps correct")


On a problem the model never saw, its states are new: pass them as `candidates` (or register them once with `model.add_states`). Expect weaker predictions here; the paper traces this limit to the state representation.


In [ ]:
model, X_te, y_te, g_te = models["extrapolation"]
problem = g_te[0]
own = sorted({s for (s, _), g in zip(X_te, g_te) if g == problem} | {n for n, g in zip(y_te, g_te) if g == problem})
top5 = model.predict_topk([X_te[0]], k=5, candidates=own)[0]
print(f"{len(own)} candidate states in this new problem; the true next state is in the top 5: {y_te[0] in top5}")


## 4. Your own data

Give EmbedPlan trajectories: the states a run went through and the actions between them, all as text. Any domain works: planning problems, game logs, web or UI agents, lab protocols.

In [ ]:
trajectories = [
    (["The door is closed. The light is off.", "The door is open. The light is off.", "The door is open. The light is on."],
     ["open the door", "switch on the light"]),
    (["The door is open. The light is on.", "The door is open. The light is off.", "The door is closed. The light is off."],
     ["switch off the light", "close the door"]),
]
X, y, groups = transitions_from_trajectories(trajectories)
tiny = EmbedPlan(encoder="hashing", max_epochs=100).fit(X, y)
print(tiny.predict([("The door is closed. The light is off.", "open the door")]))

## 5. A stronger encoder

The paper encodes states with frozen language models from MPNet to Llama-3.3-70B. Any sentence-transformers or Hugging Face model name works, or your own function from texts to vectors (for example a lookup into embeddings you already computed). A GPU runtime helps.

```python
%pip install -q "embedplan[encoders] @ git+https://github.com/embedplan/EmbedPlan.git"
model = EmbedPlan(encoder="BAAI/bge-m3").fit(X_tr, y_tr, groups=g_tr)
```

`EmbedPlan(**EmbedPlan.paper_params())` uses the hyperparameters behind the paper's main tables. Save with `model.save("model.pt")` and load with `EmbedPlan.load("model.pt")`.

## Cite

```bibtex
@article{shlomi2026textual,
  title   = {Textual Planning with Explicit Latent Transitions},
  author  = {Shlomi, Eliezer and Levy, Ido and Shapira, Eilam and Katz, Michael and Uziel, Guy
             and Shlomov, Segev and Mashkif, Nir and Reichart, Roi and Keren, Sarah},
  journal = {arXiv preprint arXiv:2602.04557},
  year    = {2026}
}
```